In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

d    = np.load('/content/oasis1_slices_1ch_all436.npz')
IMGS = d['imgs']
pos  = {sid: i for i, sid in enumerate(d['ids'])}

df = pd.read_excel('/content/oasis_cross-sectional-5708aa0a98d82080.xlsx')
df = df[df['ID'].isin(pos)].copy()
df['img_idx'] = df['ID'].map(pos)
df['subject'] = df['ID'].str.extract(r'(OAS1_\d{4})')[0]
to_label = lambda c: 0 if (pd.isna(c) or c == 0) else (1 if c == 0.5 else 2)
df['label'] = df['CDR'].apply(to_label)

A = df.reset_index(drop=True)
B = df[df['CDR'].notna()].reset_index(drop=True)

cvA = list(StratifiedGroupKFold(5, shuffle=True, random_state=42).split(A, A['label'], groups=A['subject']))
cvB = list(StratifiedKFold(5, shuffle=True, random_state=42).split(B, B['label']))

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([transforms.ToPILImage(), transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=7, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([transforms.ToPILImage(), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class DS(Dataset):
    def __init__(self, img_idx, y, tf): self.i, self.y, self.tf = img_idx, y, tf
    def __len__(self): return len(self.i)
    def __getitem__(self, k):
        img = np.repeat(IMGS[self.i[k]][:, :, None], 3, axis=2)
        return self.tf(img), torch.tensor(int(self.y[k]))

def predict(m, img_idx):
    dl = DataLoader(DS(img_idx, np.zeros(len(img_idx), int), eval_tf), batch_size=32)
    with torch.no_grad():
        return np.concatenate([torch.softmax(m(x.to(device)), 1).cpu().numpy() for x, _ in dl])

print(device, IMGS.shape, 'B:', len(B), np.bincount(B['label']))

In [ ]:
def train_variant(img_idx, y, ncls, weighting, smoothing, seed=42):
    torch.manual_seed(seed)
    dl = DataLoader(DS(img_idx, y, train_tf), batch_size=16, shuffle=True, drop_last=True)
    m = models.resnet18(weights='IMAGENET1K_V1')
    m.fc = nn.Linear(m.fc.in_features, ncls); m = m.to(device)
    cnt = np.bincount(y, minlength=ncls)
    if weighting == 'full':   w = len(y) / (ncls * cnt)
    elif weighting == 'sqrt': w = np.sqrt(len(y) / (ncls * cnt))
    else:                     w = None
    w = torch.tensor(w, dtype=torch.float32, device=device) if w is not None else None
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=smoothing)
    def go(ep, o):
        s = torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=ep)
        for _ in range(ep):
            m.train()
            for x, t in dl:
                x, t = x.to(device), t.to(device)
                o.zero_grad(); crit(m(x), t).backward(); o.step()
            s.step()
    for p in m.parameters(): p.requires_grad = False
    for p in m.fc.parameters(): p.requires_grad = True
    go(25, torch.optim.AdamW(m.fc.parameters(), lr=1e-3, weight_decay=1e-4))
    for p in m.parameters(): p.requires_grad = True
    go(15, torch.optim.AdamW([
        {'params': m.layer1.parameters(), 'lr': 1e-6}, {'params': m.layer2.parameters(), 'lr': 1e-6},
        {'params': m.layer3.parameters(), 'lr': 1e-5}, {'params': m.layer4.parameters(), 'lr': 1e-5},
        {'params': m.fc.parameters(), 'lr': 1e-4}], weight_decay=1e-4))
    return m.eval()

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, recall_score
yB = B['label'].values
res = {}
for name, wt, sm in [('full', 'full', 0.0), ('sqrt', 'sqrt', 0.05), ('none', None, 0.05)]:
    P = np.zeros((len(B), 3))
    for tr, te in cvB:
        m = train_variant(B['img_idx'].values[tr], yB[tr], 3, wt, sm)
        P[te] = predict(m, B['img_idx'].values[te])
    np.save(f'/content/oof_B_3class_1ch_{name}.npy', P)
    pred = P.argmax(1)
    res[name] = (accuracy_score(yB, pred), f1_score(yB, pred, average='macro'),
                 roc_auc_score(yB, P, multi_class='ovr', average='macro'), recall_score(yB, pred, average=None)[2])
    print(f'{name:<5} acc {res[name][0]:.3f}  macroF1 {res[name][1]:.3f}  AUC {res[name][2]:.3f}  AD recall {res[name][3]:.3f}')

accs = [r[0] for r in res.values()]; aucs = [r[2] for r in res.values()]
print(f'\naccuracy range: {(max(accs)-min(accs))*100:.1f} points   AUC range: {max(aucs)-min(aucs):.3f}')